# insurance-distributional

**Distributional GBM for insurance pricing: per-risk coefficient of variation, not just E[Y|x].**

This notebook shows the core workflow in under two minutes: fit a `TweedieGBM` on synthetic motor data, inspect the per-risk mean and variance predictions, and use the volatility score for safety-loaded pricing.

The problem this solves: two risks priced at the same pure premium can have very different loss volatility. A standard GBM treats them identically. Distributional GBM gives you both `pred.mean` (the price) and `pred.volatility_score()` (the uncertainty) - so safety loadings, underwriter referrals, and IFRS 17 risk adjustments can vary by risk.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-distributional/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-distributional catboost polars

## Synthetic data

We build 2,000 synthetic motor policies where the Tweedie dispersion varies by covariate: older vehicles have higher phi (more volatile losses). This is the key feature that a standard GBM with scalar dispersion misses.

In [ ]:
import numpy as np
import polars as pl

rng = np.random.default_rng(42)
n = 2_000

vehicle_age  = rng.integers(0, 15, n).astype(float)
driver_age   = rng.integers(18, 70, n).astype(float)
ncd_years    = rng.integers(0, 6, n).astype(float)
vehicle_group = rng.integers(1, 6, n).astype(float)  # 1=small, 5=prestige

X = np.column_stack([vehicle_age, driver_age, ncd_years, vehicle_group])
exposure = rng.uniform(0.5, 1.0, n)

# True log-linear mean: young driver penalty, NCD discount, area loading
log_mu = (
    5.0
    + 0.025 * np.maximum(30 - driver_age, 0)  # young driver penalty
    + 0.04 * vehicle_age                        # older vehicles cost more
    - 0.10 * ncd_years                          # NCD discount
    + 0.08 * vehicle_group                      # prestige loading
)
mu_true  = np.exp(log_mu)
phi_true = 0.5 + 0.03 * vehicle_age  # KEY: older vehicles are more dispersed

# Simulate Tweedie (compound Poisson-Gamma) with Tweedie power p=1.5
p = 1.5
lam = mu_true ** (2 - p) / (phi_true * (2 - p))
counts = rng.poisson(lam * exposure)
gamma_scale = mu_true / lam
y = np.array([
    float(rng.gamma(c, gamma_scale[i])) if c > 0 else 0.0
    for i, c in enumerate(counts)
])

n_train = int(0.8 * n)
X_train, X_test = X[:n_train], X[n_train:]
y_train, y_test = y[:n_train], y[n_train:]
exp_train, exp_test = exposure[:n_train], exposure[n_train:]
phi_test = phi_true[n_train:]

print(f"Training: {n_train:,}  |  Test: {n - n_train:,}  |  Zero-loss policies: {(y == 0).mean():.1%}")

## Fit TweedieGBM

`TweedieGBM` fits two CatBoost models: one for mu (the mean, using Tweedie log-likelihood) and one for phi (the dispersion, using squared Pearson residuals per Smyth & Jorgensen 2002). The dispersion model is what separates this from a standard GBM.

In [ ]:
from insurance_distributional import TweedieGBM

model = TweedieGBM(
    power=1.5,
    catboost_params_mu={"iterations": 200, "learning_rate": 0.05, "depth": 5, "verbose": 0},
    catboost_params_phi={"iterations": 100, "learning_rate": 0.05, "depth": 4, "verbose": 0},
)
model.fit(X_train, y_train, exposure=exp_train)

pred = model.predict(X_test, exposure=exp_test)

print(f"Mean prediction range: [{pred.mean.min():.1f}, {pred.mean.max():.1f}]")
print(f"Dispersion (phi) range: [{pred.phi.min():.3f}, {pred.phi.max():.3f}]")
print(f"Volatility score (CoV) range: [{pred.volatility_score().min():.3f}, {pred.volatility_score().max():.3f}]")

## Per-risk mean and volatility

The core value: `pred.mean` is the standard pure premium, `pred.volatility_score()` is the coefficient of variation (SD/mean) per risk. Two risks at similar price levels can have very different CoV depending on vehicle age.

In [ ]:
results = pl.DataFrame({
    "vehicle_age":  X_test[:, 0].astype(int),
    "driver_age":   X_test[:, 1].astype(int),
    "ncd_years":    X_test[:, 2].astype(int),
    "pure_premium": pred.mean.round(2),
    "phi_hat":      pred.phi.round(4),
    "phi_true":     phi_test.round(4),
    "cov":          pred.volatility_score().round(4),
    "safety_loaded_price": (pred.mean * (1 + 0.5 * pred.volatility_score())).round(2),
})

print("Sample predictions (sorted by volatility):")
print(results.sort("cov", descending=True).head(10))

## Scoring: CRPS and coverage

Standard metrics like Tweedie deviance only measure mean prediction quality. Distributional models need distributional metrics: CRPS (proper scoring rule combining accuracy and calibration) and coverage (does the 90% prediction interval actually contain 90% of outcomes?).

In [ ]:
from insurance_distributional import tweedie_deviance, coverage

dev  = tweedie_deviance(y_test, pred.mean, power=1.5)
crps = model.crps(X_test, y_test)
cov  = coverage(y_test, pred, levels=(0.80, 0.90, 0.95))

print(f"Tweedie deviance: {dev:.6f}")
print(f"CRPS:             {crps:.6f}  (lower is better)")
print()
print("Coverage (nominal vs actual):")
for level, actual in cov.items():
    print(f"  {level:.0%} interval: {actual:.1%} actual  ({'OK' if abs(actual - level) < 0.05 else 'UNDER' if actual < level else 'OVER'})")

## What you should see

The high-CoV risks are older vehicles - matching the true DGP where `phi_true = 0.5 + 0.03 * vehicle_age`. Newer vehicles (age 0-3) have CoV ~0.3-0.4; older vehicles (age 12+) have CoV ~0.6-0.8.

Safety-loaded prices use `P = mu * (1 + k * CoV)` where k=0.5 is the risk appetite. On a portfolio with scalar dispersion, every risk gets the same loading regardless of vehicle age. The distributional model concentrates additional loading where the true volatility is higher.

UK pricing teams use the volatility score for:

- **Safety loading** - differentiated by risk, not a single percentage
- **Underwriter referrals** - flag risks where CoV exceeds a threshold (e.g. 0.8)
- **IFRS 17 risk adjustment** - CoV is the input to the risk adjustment calculation
- **Reinsurance** - identify segments driving tail exposure

No commercial pricing tool (Emblem, Radar, Guidewire) currently outputs CoV per risk. This library fills that gap.

**GitHub:** https://github.com/burning-cost/insurance-distributional  
**PyPI:** https://pypi.org/project/insurance-distributional/  
**Blog:** [Your Technical Price Ignores Variance](https://burning-cost.github.io/2026/03/08/insurance-distributional/)